# 📖 Notebook 7: Versioning & Determinism — Safe Workflow Updates

When you deploy new workflow code, existing running workflows replay their history. If your code changes break replay, you get nondeterminism errors. This notebook shows how determinism works, what kinds of changes are unsafe, and how to evolve workflows safely with Temporal patching.

## Learning Objectives

- Understand why Temporal workflows must be deterministic
- List common determinism violations (random, time, I/O, threads)
- Use `workflow.patched()` to safely evolve workflow logic
- Use `workflow.deprecate_patch()` to clean up old branches
- Test versioning with replay tests


## 🛠️ Setup

Make sure Temporal is running:
```bash
cd 03-technologies/workflow-engines/temporal
docker compose up -d
```

### Kernel Selection
Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → 'Reload Window'.


In [ ]:
import asyncio
import uuid
from datetime import timedelta
from dataclasses import dataclass
from temporalio import activity, workflow
from temporalio.client import Client
from temporalio.worker import Worker
from temporalio.common import RetryPolicy

client = await Client.connect("localhost:7233")
TASK_QUEUE = "advanced-task-queue"
print("✅ Connected to Temporal")


---
## 🎯 What is Determinism?

Temporal replays your workflow code from the beginning every time a worker restarts or picks up a workflow task. The replayed code must produce the **exact same sequence of commands** as before: the same activities, the same timers, and the same child workflow calls in the same order.

If the code and the recorded history disagree, Temporal raises a **NonDeterminismError** because it can no longer trust the workflow state.

```text
Event History:                    Your Code Must Produce:
  1. ActivityScheduled(A)    →    execute_activity(A) ✅
  2. ActivityCompleted(A)
  3. TimerStarted(5s)        →    workflow.sleep(5)    ✅
  4. TimerFired
  5. ActivityScheduled(B)    →    execute_activity(B) ✅
```


### Unsafe changes

| Change | Why It Breaks |
|--------|--------------|
| Reorder activities | History says A then B, code says B then A |
| Remove a timer | History has timer event, code skips it |
| Add activity before existing one | History says activity X at position 3, code puts Y at position 3 |
| Use random() | Different value on replay |
| Use datetime.now() | Different time on replay |
| Do HTTP calls in workflow | Side effect produces different result on replay |


### Practical Exercise: build a simple v1 workflow

We'll create a workflow with two activities in a fixed order. This is our safe starting point. Later, we'll imagine changing that order and explain why that would be dangerous for already-running workflows.


In [ ]:
@activity.defn
async def reserve_inventory(order_id: str) -> str:
    activity.logger.info(f"Reserving inventory for {order_id}")
    await asyncio.sleep(0.5)
    return "inventory-reserved"


@activity.defn
async def charge_card(order_id: str) -> str:
    activity.logger.info(f"Charging card for {order_id}")
    await asyncio.sleep(0.5)
    return "card-charged"


@workflow.defn
class PaymentWorkflowV1:
    @workflow.run
    async def run(self, order_id: str) -> str:
        inventory_result = await workflow.execute_activity(
            reserve_inventory,
            order_id,
            start_to_close_timeout=timedelta(seconds=10),
        )
        payment_result = await workflow.execute_activity(
            charge_card,
            order_id,
            start_to_close_timeout=timedelta(seconds=10),
        )
        return f"{order_id}: {inventory_result} → {payment_result}"


print("✅ Defined PaymentWorkflowV1")


In [ ]:
# NOTE: Temporal normally runs workflow code in a sandbox that re-imports the
# defining module. In a notebook that module is `__main__`, so the re-import
# re-runs these cells and fails with "Failed validating workflow ..." (caused by
# "asyncio.run() cannot be called from a running event loop"). Notebooks must use
# UnsandboxedWorkflowRunner. In a real worker process keep the default sandbox --
# it is what protects you from non-deterministic imports.
from temporalio.worker import UnsandboxedWorkflowRunner
async def run_v1_workflow() -> str:
    async with Worker(
        client,
        task_queue=TASK_QUEUE,
        workflows=[PaymentWorkflowV1],
        activities=[reserve_inventory, charge_card],
        workflow_runner=UnsandboxedWorkflowRunner(),
    ):
        return await client.execute_workflow(
            PaymentWorkflowV1.run,
            "order-v1",
            id=f"payment-v1-{uuid.uuid4()}",
            task_queue=TASK_QUEUE,
        )


v1_result = await run_v1_workflow()
print(f"📦 V1 result: {v1_result}")


### Unsafe change example

Imagine changing the workflow so it runs `charge_card()` **before** `reserve_inventory()`.

That looks harmless in source code, but it is unsafe for running workflows. Their history says the first command was `reserve_inventory`. On replay, the new code would try to schedule `charge_card` first. That mismatch is exactly what nondeterminism means.

```python
# Unsafe idea for in-flight workflows
@workflow.defn
class PaymentWorkflowBroken:
    @workflow.run
    async def run(self, order_id: str) -> str:
        await workflow.execute_activity(charge_card, order_id, start_to_close_timeout=timedelta(seconds=10))
        await workflow.execute_activity(reserve_inventory, order_id, start_to_close_timeout=timedelta(seconds=10))
        return "broken on replay"
```

We won't run this version. The goal is to understand *why* it would fail.


---
## 🩹 Safe evolution with `workflow.patched()`

A patch is like a permanent feature flag stored in workflow history. New workflow runs record the patch marker and take the new branch. Old workflow runs do not have that marker, so they keep taking the old branch during replay.

That is how you change workflow logic without breaking executions that are already in progress.


In [ ]:
@activity.defn
async def fraud_check(order_id: str) -> str:
    activity.logger.info(f"Running fraud check for {order_id}")
    await asyncio.sleep(0.5)
    return "fraud-check-passed"


@activity.defn
async def process_payment(order_id: str) -> str:
    activity.logger.info(f"Processing payment for {order_id}")
    await asyncio.sleep(0.5)
    return "payment-processed"


@activity.defn
async def send_notification(order_id: str) -> str:
    activity.logger.info(f"Sending notification for {order_id}")
    await asyncio.sleep(0.5)
    return "notification-sent"


print("✅ Defined activities: fraud_check, process_payment, send_notification")


In [ ]:
@workflow.defn(name="PaymentWorkflow")
class PaymentWorkflow:
    @workflow.run
    async def run(self, order_id: str) -> str:
        if workflow.patched("add-fraud-check"):
            fraud_result = await workflow.execute_activity(
                fraud_check,
                order_id,
                start_to_close_timeout=timedelta(seconds=10),
            )
        else:
            fraud_result = "fraud-check-skipped-for-old-history"

        payment_result = await workflow.execute_activity(
            process_payment,
            order_id,
            start_to_close_timeout=timedelta(seconds=10),
        )
        notification_result = await workflow.execute_activity(
            send_notification,
            order_id,
            start_to_close_timeout=timedelta(seconds=10),
        )
        return f"{order_id}: {fraud_result} → {payment_result} → {notification_result}"


print("✅ Defined patched PaymentWorkflow")


In [ ]:
# NOTE: Temporal normally runs workflow code in a sandbox that re-imports the
# defining module. In a notebook that module is `__main__`, so the re-import
# re-runs these cells and fails with "Failed validating workflow ..." (caused by
# "asyncio.run() cannot be called from a running event loop"). Notebooks must use
# UnsandboxedWorkflowRunner. In a real worker process keep the default sandbox --
# it is what protects you from non-deterministic imports.
from temporalio.worker import UnsandboxedWorkflowRunner
async def run_patched_workflow() -> str:
    async with Worker(
        client,
        task_queue=TASK_QUEUE,
        workflows=[PaymentWorkflow],
        activities=[fraud_check, process_payment, send_notification],
        workflow_runner=UnsandboxedWorkflowRunner(),
    ):
        return await client.execute_workflow(
            PaymentWorkflow.run,
            "order-patched",
            id=f"payment-patched-{uuid.uuid4()}",
            task_queue=TASK_QUEUE,
        )


patched_result = await run_patched_workflow()
print(f"🩹 Patched workflow result: {patched_result}")


### How patch markers work

`workflow.patched("add-fraud-check")` writes a marker into history.

- **Old workflows** have no marker, so they keep taking the old branch during replay
- **New workflows** record the marker and take the new branch

### Three-phase deployment lifecycle

1. **Phase 1: Add `patched()`** — old and new paths both exist
2. **Phase 2: `deprecate_patch()`** — only the new path runs, but the marker is still recognized
3. **Phase 3: Remove patch code** — after all old workflows are gone


### Practical Exercise: deprecating a patch

Once you're sure all old histories are gone, you can remove the old branch and keep only the new path. `workflow.deprecate_patch()` is the transition step that makes that safe.


In [ ]:
@workflow.defn(name="PaymentWorkflowV3")
class PaymentWorkflowV3:
    @workflow.run
    async def run(self, order_id: str) -> str:
        workflow.deprecate_patch("add-fraud-check")
        fraud_result = await workflow.execute_activity(
            fraud_check,
            order_id,
            start_to_close_timeout=timedelta(seconds=10),
        )
        payment_result = await workflow.execute_activity(
            process_payment,
            order_id,
            start_to_close_timeout=timedelta(seconds=10),
        )
        notification_result = await workflow.execute_activity(
            send_notification,
            order_id,
            start_to_close_timeout=timedelta(seconds=10),
        )
        return f"{order_id}: {fraud_result} → {payment_result} → {notification_result}"


print("✅ Defined PaymentWorkflowV3")


In [ ]:
# NOTE: Temporal normally runs workflow code in a sandbox that re-imports the
# defining module. In a notebook that module is `__main__`, so the re-import
# re-runs these cells and fails with "Failed validating workflow ..." (caused by
# "asyncio.run() cannot be called from a running event loop"). Notebooks must use
# UnsandboxedWorkflowRunner. In a real worker process keep the default sandbox --
# it is what protects you from non-deterministic imports.
from temporalio.worker import UnsandboxedWorkflowRunner
async def run_v3_workflow() -> str:
    async with Worker(
        client,
        task_queue=TASK_QUEUE,
        workflows=[PaymentWorkflowV3],
        activities=[fraud_check, process_payment, send_notification],
        workflow_runner=UnsandboxedWorkflowRunner(),
    ):
        return await client.execute_workflow(
            PaymentWorkflowV3.run,
            "order-v3",
            id=f"payment-v3-{uuid.uuid4()}",
            task_queue=TASK_QUEUE,
        )


v3_result = await run_v3_workflow()
print(f"🚀 V3 result: {v3_result}")


---
## ✅ Safe Workflow Code Rules Checklist

- ✅ Use `workflow.execute_activity()` for side effects
- ✅ Use `asyncio.sleep()` (Temporal-safe in Python SDK)
- ✅ Use `workflow.patched()` for code changes
- ✅ Use `workflow.uuid4()` for random IDs when your SDK version exposes it; otherwise prefer deterministic helpers or pass IDs in from outside the workflow
- ❌ Don't use `random.random()`
- ❌ Don't use `datetime.now()` — use `workflow.now()`
- ❌ Don't do HTTP/DB calls in workflow code
- ❌ Don't use threading


### Practical Exercise: replay testing

Replay testing is how you prove a workflow change is still compatible with real history. Export a workflow history from the Temporal UI or CLI, then replay it against your current workflow code. If replay fails, your change is not safe.


In [ ]:
from temporalio.client import WorkflowHistory
from temporalio.worker import Replayer


async def replay_payment_history(history_json: str) -> None:
    replayer = Replayer(workflows=[PaymentWorkflow, PaymentWorkflowV3])
    await replayer.replay_workflow(WorkflowHistory.from_json(history_json))


print("✅ Defined replay helper")
print("   Export a workflow history as JSON, load it into a string,")
print("   then call: await replay_payment_history(history_json)")


## 🎓 What You Learned

- Temporal workflows must be **deterministic** because the worker replays history to rebuild state
- Reordering commands, removing timers, or adding side effects inside workflow code can break replay
- `workflow.patched()` is the safe way to evolve running workflows
- `workflow.deprecate_patch()` is the cleanup step before fully removing patch logic
- Replay tests let you validate real workflow histories before you deploy new worker code

### Suggested next experiments

1. Add a second patch to `PaymentWorkflow`
2. Export a real history from Temporal and replay it with `Replayer`
3. Replace an activity timeout and think through whether it changes command order
4. Compare Worker Versioning and patching in the Temporal docs
